# Module B: Executive Power BI Data Preparation & Dimensional Modeling
## RISK Enterprise Decision Intelligence Platform
**Role**: Power BI Solutions Architect & Data Engineer  
**Objective**: Transform cleaned retail transactional records into an enterprise-grade Star Schema architecture (Kimball methodology), optimizing relationship cardinality, DAX query performance, and time intelligence.


### 1. Dimensional Architecture Blueprint
The model decomposes the flat 21-column dataset into:
- **`FactSales`**: Grain of one row per line item ordered (`Row ID`).
- **`DimDate`**: Continuous calendar table supporting DAX Time Intelligence.
- **`DimProduct`**: Product master hierarchy (`Category` -> `Sub-Category` -> `Product Name`).
- **`DimCustomer`**: Customer master hierarchy (`Segment` -> `Customer Name`).
- **`DimGeography`**: Geography hierarchy (`Country` -> `Region` -> `State` -> `City` -> `Postal Code`).

All relationships are **1-to-Many single direction (1:*)** from Dimensions to Fact, strictly preventing ambiguous many-to-many relationships.


In [ ]:
import pandas as pd
import numpy as np

# Load the generated dimensional tables
fact_sales = pd.read_csv('../data/processed/powerbi_fact_sales.csv')
dim_date = pd.read_csv('../data/processed/powerbi_dim_date.csv')
dim_prod = pd.read_csv('../data/processed/powerbi_dim_product.csv')
dim_cust = pd.read_csv('../data/processed/powerbi_dim_customer.csv')
dim_geo = pd.read_csv('../data/processed/powerbi_dim_geography.csv')

print(f"FactSales:   {fact_sales.shape[0]:,} rows, {fact_sales.shape[1]} cols")
print(f"DimDate:     {dim_date.shape[0]:,} rows, {dim_date.shape[1]} cols")
print(f"DimProduct:  {dim_prod.shape[0]:,} rows, {dim_prod.shape[1]} cols")
print(f"DimCustomer: {dim_cust.shape[0]:,} rows, {dim_cust.shape[1]} cols")
print(f"DimGeography:{dim_geo.shape[0]:,} rows, {dim_geo.shape[1]} cols")


### 2. Referential Integrity & Foreign Key Coverage Audit
Asserting that 100% of foreign keys in `FactSales` map uniquely and cleanly to their respective Dimension primary keys.


In [ ]:
assert fact_sales['CustomerKey'].isin(dim_cust['CustomerKey']).all(), 'CustomerKey foreign key violation!'
assert fact_sales['ProductKey'].isin(dim_prod['ProductKey']).all(), 'ProductKey foreign key violation!'
assert fact_sales['GeoKey'].isin(dim_geo['GeoKey']).all(), 'GeoKey foreign key violation!'
assert fact_sales['OrderDateKey'].isin(dim_date['DateKey']).all(), 'OrderDateKey foreign key violation!'
assert fact_sales['ShipDateKey'].isin(dim_date['DateKey']).all(), 'ShipDateKey foreign key violation!'
print('Referential integrity: 100% verified across all dimensional relationships.')


### 3. Dimensional Metric Reconciliation
Confirming that aggregations on `FactSales` reconcile exactly with the master Python numbers.


In [ ]:
print(f"FactSales Total Revenue: ${fact_sales['Sales'].sum():,.2f}")
print(f"FactSales Total Profit:  ${fact_sales['Profit'].sum():,.2f}")
print(f"FactSales Total Orders:  {fact_sales['Order ID'].nunique():,}")
print(f"FactSales Total Volume:  {fact_sales['Quantity'].sum():,} units")
